In [17]:
import os
import sys

# Add the project root to the system path so Python can find 'src'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

import torch
import torchaudio
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from src.audio.constants import *
from src.audio.logger import logging
from src.audio.exception import CustomException
from src.audio.entity.config_entity import *
from src.audio.entity.artifact_entity import *

In [18]:
logging.getLogger("PIL").setLevel(logging.WARNING)  # Suppress PIL warnings

In [19]:
import os
import sys
import shutil
import wave
from pathlib import Path

# Add the project root to the system path so Python can find 'src'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

import numpy as np
import torch
import torchaudio
from src.audio.constants import *
from src.audio.logger import logging
from src.audio.exception import CustomException
from src.audio.entity.config_entity import *
from src.audio.entity.artifact_entity import *

class DataTransformation:
    def __init__(self, data_transformation_config: DataTransformationConfig,
                 data_ingestion_artifact: DataIngestionArtifacts):
        try:
            self.data_transformation_config = data_transformation_config
            self.data_ingestion_artifact = data_ingestion_artifact
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        except Exception as e:
            raise CustomException(e, sys)

    def _read_audio(self, file_path: str):
        try:
            import soundfile as sf
            data, sample_rate = sf.read(file_path, dtype="float32", always_2d=True)
            waveform = torch.from_numpy(data.T.copy())
            return waveform, sample_rate
        except ImportError:
            with wave.open(file_path, "rb") as wf:
                sample_rate = wf.getframerate()
                n_channels = wf.getnchannels()
                sampwidth = wf.getsampwidth()
                raw_bytes = wf.readframes(wf.getnframes())
            if sampwidth == 2:
                audio_np = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
            elif sampwidth == 4:
                audio_np = np.frombuffer(raw_bytes, dtype=np.int32).astype(np.float32) / 2147483648.0
            else:
                audio_np = (np.frombuffer(raw_bytes, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
            waveform = torch.from_numpy(audio_np.reshape(-1, n_channels).T.copy())
            return waveform, sample_rate

    def load_audio_files(self, path: str, label: str):
        try:
            dataset = []
            audio_extensions = {".wav", ".mp3", ".flac", ".ogg"}
            walker = sorted(
                str(p) for p in Path(path).rglob("*")
                if p.suffix.lower() in audio_extensions
            )
            if not walker:
                logging.warning(f"No audio files found in specified path: {path}")

            for i, file_path in enumerate(walker):
                _, filename = os.path.split(file_path)
                speaker, _ = os.path.splitext(filename)

                if "_nohash_" in speaker:
                    try:
                        speaker_id, utterance_number = speaker.split("_nohash_")
                        utterance_number = int(utterance_number)
                    except ValueError:
                        speaker_id, utterance_number = speaker, i
                else:
                    speaker_id, utterance_number = speaker, i

                waveform, sample_rate = self._read_audio(file_path)
                dataset.append((waveform, sample_rate, label, speaker_id, utterance_number))

            return dataset
        except Exception as e:
            raise CustomException(e, sys)

# 1. Setup paths
config = DataIngestionConfig()
data_root = os.path.join(config.unzip_data_dir, "data")
data_ingestion_artifact = DataIngestionArtifacts(data_folder_path=data_root)

# 2. Copy your real 'cat' and 'dog' audio folders into data_root if not already there
source_data_dir = r"C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio Binary Classification\Code\data (1)"
if os.path.exists(source_data_dir):
    for folder_name in ["cat", "dog"]:
        src_folder = os.path.join(source_data_dir, folder_name)
        dst_folder = os.path.join(data_root, folder_name)
        if os.path.exists(src_folder) and not os.path.exists(dst_folder):
            shutil.copytree(src_folder, dst_folder)

# 3. Initialize the transformer and load all audio files
transformer = DataTransformation(DataTransformationConfig, data_ingestion_artifact)

for label in os.listdir(data_root):
    label_path = os.path.join(data_root, label)
    if os.path.isdir(label_path):
        dataset = transformer.load_audio_files(label_path, label)
        print(f"Loaded {len(dataset)} files for label '{label}' from {label_path}")

Loaded 5 files for label '0' from C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data\0
Loaded 5 files for label '1' from C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data\1


In [20]:
import os

config = DataIngestionConfig()
data_root = os.path.join(config.unzip_data_dir, "data")
print("Data root:", data_root)
for root, dirs, files in os.walk(data_root):
    print("Folder:", root, "| Files count:", len(files), "| Sample files:", files[:3])

Data root: C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data
Folder: C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data | Files count: 0 | Sample files: []
Folder: C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data\0 | Files count: 5 | Sample files: ['sample_0.wav', 'sample_1.wav', 'sample_2.wav']
Folder: C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data\1 | Files count: 5 | Sample files: ['sample_0.wav', 'sample_1

In [21]:
config = DataIngestionConfig()
data_root = os.path.join(config.unzip_data_dir, "data")
data_ingestion_artifact = DataIngestionArtifacts(data_folder_path=data_root)

# Initialize the transformer
transformer = DataTransformation(DataTransformationConfig, data_ingestion_artifact)

# Loop through each label folder and load audio files
for label in os.listdir(data_root):
    label_path = os.path.join(data_root, label)
    if os.path.isdir(label_path):
        dataset = transformer.load_audio_files(label_path, label)
        print(f"Loaded {len(dataset)} files for label '{label}' from {label_path}")

Loaded 5 files for label '0' from C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data\0
Loaded 5 files for label '1' from C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data\1


In [22]:
import os
import sys
import wave
from pathlib import Path

# Add the project root to the system path so Python can find 'src'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
from src.audio.constants import *
from src.audio.logger import logging
from src.audio.exception import CustomException
from src.audio.entity.config_entity import *
from src.audio.entity.artifact_entity import *


class DataTransformation:
    def __init__(self, data_transformation_config: DataTransformationConfig,
                 data_ingestion_artifact: DataIngestionArtifacts):
        try:
            self.data_transformation_config = data_transformation_config
            self.data_ingestion_artifact = data_ingestion_artifact
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        except Exception as e:
            raise CustomException(e, sys)

    def _read_audio(self, file_path: str):
        try:
            import soundfile as sf
            data, sample_rate = sf.read(file_path, dtype="float32", always_2d=True)
            waveform = torch.from_numpy(data.T.copy())
            return waveform, sample_rate
        except ImportError:
            with wave.open(file_path, "rb") as wf:
                sample_rate = wf.getframerate()
                n_channels = wf.getnchannels()
                sampwidth = wf.getsampwidth()
                raw_bytes = wf.readframes(wf.getnframes())
            if sampwidth == 2:
                audio_np = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
            elif sampwidth == 4:
                audio_np = np.frombuffer(raw_bytes, dtype=np.int32).astype(np.float32) / 2147483648.0
            else:
                audio_np = (np.frombuffer(raw_bytes, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
            waveform = torch.from_numpy(audio_np.reshape(-1, n_channels).T.copy())
            return waveform, sample_rate

    def load_audio_files(self, path: str, label: str):
        try:
            dataset = []
            audio_extensions = {".wav", ".mp3", ".flac", ".ogg"}
            walker = sorted(
                str(p) for p in Path(path).rglob("*")
                if p.suffix.lower() in audio_extensions
            )
            if not walker:
                logging.warning(f"No audio files found in specified path: {path}")

            for i, file_path in enumerate(walker):
                _, filename = os.path.split(file_path)
                speaker, _ = os.path.splitext(filename)

                if "_nohash_" in speaker:
                    try:
                        speaker_id, utterance_number = speaker.split("_nohash_")
                        utterance_number = int(utterance_number)
                    except ValueError:
                        speaker_id, utterance_number = speaker, i
                else:
                    speaker_id, utterance_number = speaker, i

                waveform, sample_rate = self._read_audio(file_path)
                dataset.append((waveform, sample_rate, label, speaker_id, utterance_number))

            return dataset
        except Exception as e:
            raise CustomException(e, sys)

    def create_spectrogram_images(self, dataloader, label_dir, is_test=False):
        try:
            base_dir = (
                self.data_transformation_config.test_dir
                if is_test
                else self.data_transformation_config.train_dir
            )
            directory = os.path.join(base_dir, label_dir)
            if os.path.isdir(directory):
                logging.info("Spectrogram Directory Exists")
            else:
                os.makedirs(directory, exist_ok=True)

            spectrogram_count = 0
            for i, data in enumerate(dataloader):
                waveform = data[0].to(self.device)
                sample_rate = data[1]
                label = data[2]
                ID = data[3]

                spectrogram_transform = torchaudio.transforms.Spectrogram(
                    n_fft=400, win_length=None, hop_length=None, window_fn=torch.hann_window
                ).to(self.device)

                spectrogram_tensor = spectrogram_transform(waveform)
                spectrogram_data = spectrogram_tensor[0].log2()
                spectrogram_data = torch.where(
                    torch.isfinite(spectrogram_data),
                    spectrogram_data,
                    torch.tensor(0.0, device=self.device),
                )
                path_to_save_img = os.path.join(directory, f"spec_img{i}.png")
                plt.imsave(path_to_save_img, spectrogram_data.squeeze().cpu().numpy(), cmap="viridis")
                spectrogram_count += 1

            logging.info(f"Saved {spectrogram_count} spectrogram images to {directory}")
        except Exception as e:
            raise CustomException(e, sys)

In [24]:
import os
import sys
import wave
from pathlib import Path

# Add the project root to the system path so Python can find 'src'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
from src.audio.constants import *
from src.audio.logger import logging
from src.audio.exception import CustomException
from src.audio.entity.config_entity import *
from src.audio.entity.artifact_entity import *


class DataTransformation:
    def __init__(self, data_transformation_config: DataTransformationConfig,
                 data_ingestion_artifact: DataIngestionArtifacts):
        try:
            self.data_transformation_config = data_transformation_config
            self.data_ingestion_artifact = data_ingestion_artifact
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        except Exception as e:
            raise CustomException(e, sys)

    def _read_audio(self, file_path: str):
        try:
            import soundfile as sf
            data, sample_rate = sf.read(file_path, dtype="float32", always_2d=True)
            waveform = torch.from_numpy(data.T.copy())
            return waveform, sample_rate
        except ImportError:
            with wave.open(file_path, "rb") as wf:
                sample_rate = wf.getframerate()
                n_channels = wf.getnchannels()
                sampwidth = wf.getsampwidth()
                raw_bytes = wf.readframes(wf.getnframes())
            if sampwidth == 2:
                audio_np = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
            elif sampwidth == 4:
                audio_np = np.frombuffer(raw_bytes, dtype=np.int32).astype(np.float32) / 2147483648.0
            else:
                audio_np = (np.frombuffer(raw_bytes, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
            waveform = torch.from_numpy(audio_np.reshape(-1, n_channels).T.copy())
            return waveform, sample_rate

    def load_audio_files(self, path: str, label: str):
        try:
            dataset = []
            audio_extensions = {".wav", ".mp3", ".flac", ".ogg"}
            walker = sorted(
                str(p) for p in Path(path).rglob("*")
                if p.suffix.lower() in audio_extensions
            )
            if not walker:
                logging.warning(f"No audio files found in specified path: {path}")

            for i, file_path in enumerate(walker):
                _, filename = os.path.split(file_path)
                speaker, _ = os.path.splitext(filename)

                if "_nohash_" in speaker:
                    try:
                        speaker_id, utterance_number = speaker.split("_nohash_")
                        utterance_number = int(utterance_number)
                    except ValueError:
                        speaker_id, utterance_number = speaker, i
                else:
                    speaker_id, utterance_number = speaker, i

                waveform, sample_rate = self._read_audio(file_path)
                dataset.append((waveform, sample_rate, label, speaker_id, utterance_number))

            return dataset
        except Exception as e:
            raise CustomException(e, sys)

    def create_spectrogram_images(self, dataloader, label_dir, is_test=False):
        try:
            base_dir = (
                self.data_transformation_config.test_dir
                if is_test
                else self.data_transformation_config.images_dir
            )
            directory = os.path.join(base_dir, label_dir)
            if os.path.isdir(directory):
                logging.info("Spectrogram Directory Exists")
            else:
                os.makedirs(directory, exist_ok=True)

            spectrogram_count = 0
            for i, data in enumerate(dataloader):
                waveform = data[0].to(self.device)
                sample_rate = data[1]
                label = data[2]
                ID = data[3]

                spectrogram_transform = torchaudio.transforms.Spectrogram(
                    n_fft=400, win_length=None, hop_length=None, window_fn=torch.hann_window
                ).to(self.device)

                spectrogram_tensor = spectrogram_transform(waveform)
                spectrogram_data = spectrogram_tensor[0].log2()
                spectrogram_data = torch.where(
                    torch.isfinite(spectrogram_data),
                    spectrogram_data,
                    torch.tensor(0.0, device=self.device),
                )
                path_to_save_img = os.path.join(directory, f"spec_img{i}.png")
                plt.imsave(path_to_save_img, spectrogram_data.squeeze().cpu().numpy(), cmap="viridis")
                spectrogram_count += 1

            logging.info(f"Saved {spectrogram_count} spectrogram images to {directory}")
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_data_transformation(self):
        try:
            logging.info("Initiating Data Transformation")
            dog_folder_path = os.path.join(self.data_ingestion_artifact.data_folder_path, "data", "dog")
            cat_folder_path = os.path.join(self.data_ingestion_artifact.data_folder_path, "data", "cat")
            logging.info(f"dog folder exists: {os.path.exists(dog_folder_path)} ({dog_folder_path})")
            logging.info(f"cat folder exists: {os.path.exists(cat_folder_path)} ({cat_folder_path})")
            logging.info(f"Training Images Directory: {self.data_transformation_config.images_dir}")
            logging.info(f"Testing Images Directory: {self.data_transformation_config.test_dir}")

            dog_dataset = self.load_audio_files(dog_folder_path, "dog")
            cat_dataset = self.load_audio_files(cat_folder_path, "cat")
            logging.info(f"length of dog dataset: {len(dog_dataset)}")
            logging.info(f"length of cat dataset: {len(cat_dataset)}")

            if len(dog_dataset) == 0 or len(cat_dataset) == 0:
                raise CustomException("No audio files in dog or cat folders", sys)

            trainloader_dog = torch.utils.data.DataLoader(train_dog, batch_size=1, shuffle=SHUFFLE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            testloader_dog = torch.utils.data.DataLoader(test_dog, batch_size=1, shuffle=SHUFFLE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            trainloader_cat = torch.utils.data.DataLoader(train_cat, batch_size=1, shuffle=SHUFFLE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            testloader_cat = torch.utils.data.DataLoader(test_cat, batch_size=1, shuffle=SHUFFLE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

            self.create_spectrogram_images(train_dog, "dog", is_test=False)
            self.create_spectrogram_images(test_dog, "dog", is_test=True)
            self.create_spectrogram_images(train_cat, "cat", is_test=False)
            self.create_spectrogram_images(test_cat, "cat", is_test=True)

            data_transformation_artifact = DataTransformationArtifacts(
                images_folder_path=self.data_transformation_config.images_dir,
                test_folder_path=self.data_transformation_config.test_dir,
            )
            logging.info("Data Transformation complete")
            return data_transformation_artifact
        except Exception as e:
            logging.error("Data Transformation failed")
            raise CustomException(e, sys)